# Negotiating the Past - Local Classification with MLX

This notebook uses **Ministral-3-14B-Instruct** running locally on Apple Silicon via MLX to classify prompts that contain references to the past.

**Hardware requirements**: Mac with Apple Silicon (M1/M2/M3/M4) and at least 16GB unified memory (32GB+ recommended for 14B model).

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
%pip install mlx-lm pandas tqdm

In [ ]:
import os
import time
import logging
import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"mlx_classification_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("mlx_classifier")

# Create results directory
os.makedirs("data/results_mlx", exist_ok=True)

print("Libraries loaded successfully")

## 2. Load Model (MLX primary, Ollama fallback)

In [ ]:
# Configuration
MODEL_ID = "mistralai/Ministral-3-14B-Instruct-2512"
OLLAMA_MODEL = "ministral"  # Fallback model name for Ollama

USE_MLX = True  # Set to False to use Ollama instead

model = None
tokenizer = None

if USE_MLX:
    try:
        from mlx_lm import load, generate
        
        print(f"Loading model: {MODEL_ID}")
        print("This may take a few minutes on first run (downloading model weights)...")
        
        # Load the model - MLX will automatically convert if needed
        model, tokenizer = load(MODEL_ID)
        
        print(f"Model loaded successfully via MLX")
        
    except Exception as e:
        print(f"MLX loading failed: {e}")
        print("Falling back to Ollama...")
        USE_MLX = False

if not USE_MLX:
    try:
        import ollama
        
        # Test Ollama connection
        ollama.list()
        print(f"Using Ollama with model: {OLLAMA_MODEL}")
        print("Make sure the model is pulled: ollama pull ministral")
        
    except Exception as e:
        raise RuntimeError(f"Neither MLX nor Ollama available: {e}")

## 3. System Prompt

Adapted from the Claude version, with added requirement for a one-sentence justification.

In [ ]:
SYSTEM_PROMPT = """You are a panel of three historians with different specializations (international history, global history, and European history). Your task is to analyze prompts and determine if they contain an implicit or explicit reference to the past.

Consider the following criteria:

1. EXPLICIT REFERENCES: Clear temporal markers (yesterday, last week, previously), historical events, periods, or figures, or mentions of things that happened in the past.

2. IMPLICIT REFERENCES: Subtle indications of past time frames, comparative language suggesting change over time, or references to completed actions or states that are no longer current.

3. CONTEXTUAL CLUES: Words implying memory, reflection, or nostalgia; verbs in past tense that indicate historical events rather than hypotheticals.

4. DOMAIN-SPECIFIC PERSPECTIVES:
   - International historian: References to international relations, treaties, wars, or cross-border interactions that occurred in the past
   - Global historian: References to world systems, long-term global trends, or cross-cultural historical developments
   - European historian: References to European historical periods, events, or figures

IMPORTANT: Your response must follow this EXACT format:
[yes/no]: [one sentence justification]

Examples:
- yes: The prompt references Napoleon Bonaparte, a historical figure from the 19th century.
- no: The prompt describes a futuristic sci-fi scene with no historical references.
- yes: The mention of "Victorian style" implicitly refers to the historical Victorian era.
"""

print("System prompt configured")
print(f"Prompt length: {len(SYSTEM_PROMPT)} characters")

## 4. Classification Functions

In [ ]:
def format_prompt_for_model(user_prompt: str) -> str:
    """Format the prompt for Ministral instruct format."""
    # Ministral uses the standard ChatML-like format
    formatted = f"""<s>[INST] {SYSTEM_PROMPT}

Analyze this prompt: {user_prompt} [/INST]"""
    return formatted


def parse_response(response: str) -> tuple[str, str]:
    """Parse the model response into classification and justification.
    
    Returns:
        tuple: (classification: 'yes'/'no'/'error', justification: str)
    """
    response = response.strip()
    
    # Try to match the expected format: "yes: justification" or "no: justification"
    match = re.match(r'^(yes|no)\s*[:\-]\s*(.+)', response, re.IGNORECASE | re.DOTALL)
    
    if match:
        classification = match.group(1).lower()
        justification = match.group(2).strip()
        # Take only the first sentence if multiple
        justification = justification.split('.')[0] + '.' if '.' in justification else justification
        return classification, justification
    
    # Fallback: check if response starts with yes/no
    response_lower = response.lower()
    if response_lower.startswith('yes'):
        return 'yes', response[3:].strip(' :-')
    elif response_lower.startswith('no'):
        return 'no', response[2:].strip(' :-')
    
    # If we can find yes or no anywhere
    if 'yes' in response_lower[:50]:
        return 'yes', response
    elif 'no' in response_lower[:50]:
        return 'no', response
    
    # Could not parse
    logger.warning(f"Could not parse response: {response[:100]}")
    return 'error', response


def classify_prompt_mlx(prompt: str, max_tokens: int = 100) -> tuple[str, str, float]:
    """Classify a single prompt using MLX.
    
    Returns:
        tuple: (classification, justification, generation_time)
    """
    formatted = format_prompt_for_model(prompt)
    
    start_time = time.time()
    
    response = generate(
        model,
        tokenizer,
        prompt=formatted,
        max_tokens=max_tokens,
        temp=0.0,  # Deterministic
    )
    
    generation_time = time.time() - start_time
    
    classification, justification = parse_response(response)
    return classification, justification, generation_time


def classify_prompt_ollama(prompt: str, max_tokens: int = 100) -> tuple[str, str, float]:
    """Classify a single prompt using Ollama (fallback).
    
    Returns:
        tuple: (classification, justification, generation_time)
    """
    import ollama
    
    start_time = time.time()
    
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f"Analyze this prompt: {prompt}"}
        ],
        options={
            'num_predict': max_tokens,
            'temperature': 0.0,
        }
    )
    
    generation_time = time.time() - start_time
    
    response_text = response['message']['content']
    classification, justification = parse_response(response_text)
    return classification, justification, generation_time


def classify_prompt(prompt: str, max_tokens: int = 100) -> tuple[str, str, float]:
    """Classify a prompt using the configured backend (MLX or Ollama)."""
    if USE_MLX:
        return classify_prompt_mlx(prompt, max_tokens)
    else:
        return classify_prompt_ollama(prompt, max_tokens)


print("Classification functions defined")

## 5. Load Dataset

In [ ]:
# Configuration for validation phase
SAMPLE_SIZE = 1000  # Start with 1000 for validation

# Load prompts
print(f"Loading {SAMPLE_SIZE} prompts for validation...")

# Count total rows
total_rows = sum(1 for _ in open('data/prompts.csv')) - 1  # -1 for header
print(f"Total prompts in dataset: {total_rows:,}")

# Compute skip probability for random sampling
skip_prob = 1 - SAMPLE_SIZE / total_rows

# Load sample
np.random.seed(42)  # For reproducibility
prompts_df = pd.read_csv(
    'data/prompts.csv',
    usecols=[0],
    skiprows=lambda x: x > 0 and np.random.random() < skip_prob
)

# Convert to list and clean
prompts_list = prompts_df.iloc[:, 0].dropna().tolist()
prompts_list = [str(p).strip() for p in prompts_list if str(p).strip()]

# Limit to exact sample size
prompts_list = prompts_list[:SAMPLE_SIZE]

print(f"Loaded {len(prompts_list)} prompts")
print("\nFirst 3 prompts:")
for i, p in enumerate(prompts_list[:3]):
    print(f"  {i+1}. {p[:80]}{'...' if len(p) > 80 else ''}")

## 6. Test Single Classification

Test the model on a single prompt to verify everything works.

In [ ]:
# Test with a single prompt
test_prompt = prompts_list[0] if prompts_list else "Napoleon Bonaparte leading his army across the Alps"

print(f"Testing with prompt: {test_prompt}")
print("="*60)

classification, justification, gen_time = classify_prompt(test_prompt)

print(f"Classification: {classification}")
print(f"Justification: {justification}")
print(f"Generation time: {gen_time:.2f}s")
print("="*60)

# Estimate time for full validation
estimated_total_time = gen_time * len(prompts_list)
print(f"\nEstimated time for {len(prompts_list)} prompts: {estimated_total_time/60:.1f} minutes")

## 7. Run Validation (1000 prompts)

Process the validation sample with progress tracking and incremental saves.

In [ ]:
# Configuration
BATCH_SIZE = 50  # Save results every N prompts
RESULTS_FILE = "data/results_mlx/validation_results.csv"

# Check for existing results (for resuming)
start_idx = 0
existing_results = []

if os.path.exists(RESULTS_FILE):
    existing_df = pd.read_csv(RESULTS_FILE)
    start_idx = len(existing_df)
    existing_results = existing_df.to_dict('records')
    print(f"Resuming from index {start_idx} ({start_idx} prompts already processed)")

# Process prompts
results = existing_results.copy()
generation_times = []

remaining_prompts = prompts_list[start_idx:]
print(f"Processing {len(remaining_prompts)} prompts...")
print("="*60)

start_time = time.time()

for i, prompt in enumerate(tqdm(remaining_prompts, desc="Classifying")):
    try:
        classification, justification, gen_time = classify_prompt(prompt)
        generation_times.append(gen_time)
        
        results.append({
            'prompt': prompt,
            'references_past': classification,
            'justification': justification,
            'generation_time': gen_time
        })
        
    except Exception as e:
        logger.error(f"Error processing prompt {start_idx + i}: {e}")
        results.append({
            'prompt': prompt,
            'references_past': 'error',
            'justification': str(e),
            'generation_time': 0
        })
    
    # Save incrementally
    if (i + 1) % BATCH_SIZE == 0:
        pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)
        avg_time = np.mean(generation_times[-BATCH_SIZE:])
        logger.info(f"Saved {len(results)} results. Avg time: {avg_time:.2f}s/prompt")

# Final save
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_FILE, index=False)

total_time = time.time() - start_time
print("="*60)
print(f"Completed! Total time: {total_time/60:.1f} minutes")
print(f"Average time per prompt: {np.mean(generation_times):.2f}s")

## 8. Validation Results Summary

In [ ]:
# Load and analyze results
results_df = pd.read_csv(RESULTS_FILE)

print("=" * 60)
print("VALIDATION RESULTS SUMMARY")
print("=" * 60)

total = len(results_df)
yes_count = (results_df['references_past'] == 'yes').sum()
no_count = (results_df['references_past'] == 'no').sum()
error_count = (results_df['references_past'] == 'error').sum()

print(f"Total prompts processed: {total}")
print(f"References to past (yes): {yes_count} ({yes_count/total*100:.2f}%)")
print(f"No reference (no): {no_count} ({no_count/total*100:.2f}%)")
print(f"Errors: {error_count} ({error_count/total*100:.2f}%)")

print("\n" + "=" * 60)
print("PERFORMANCE METRICS")
print("=" * 60)

avg_time = results_df['generation_time'].mean()
print(f"Average generation time: {avg_time:.2f}s per prompt")
print(f"Throughput: {3600/avg_time:.0f} prompts/hour")

# Estimate for full dataset
full_dataset_hours = (10_000_000 * avg_time) / 3600
print(f"\nEstimated time for 10M prompts: {full_dataset_hours:.0f} hours ({full_dataset_hours/24:.1f} days)")

In [ ]:
# Show sample results with justifications
print("\n" + "=" * 60)
print("SAMPLE RESULTS WITH JUSTIFICATIONS")
print("=" * 60)

# Show some 'yes' examples
yes_samples = results_df[results_df['references_past'] == 'yes'].head(5)
print("\n--- Prompts WITH references to the past ---")
for _, row in yes_samples.iterrows():
    print(f"\nPrompt: {row['prompt'][:100]}{'...' if len(row['prompt']) > 100 else ''}")
    print(f"Justification: {row['justification']}")

# Show some 'no' examples
no_samples = results_df[results_df['references_past'] == 'no'].head(5)
print("\n--- Prompts WITHOUT references to the past ---")
for _, row in no_samples.iterrows():
    print(f"\nPrompt: {row['prompt'][:100]}{'...' if len(row['prompt']) > 100 else ''}")
    print(f"Justification: {row['justification']}")

## 9. Compare with Claude Results (Optional)

If you have existing Claude API results, compare them here.

In [ ]:
# Optional: Compare with Claude results if available
CLAUDE_RESULTS_FILE = "data/results_claude/past_references_complete_results.csv"

if os.path.exists(CLAUDE_RESULTS_FILE):
    claude_df = pd.read_csv(CLAUDE_RESULTS_FILE)
    
    # Find matching prompts
    merged = results_df.merge(
        claude_df[['prompt', 'references_past']],
        on='prompt',
        how='inner',
        suffixes=('_mlx', '_claude')
    )
    
    if len(merged) > 0:
        agreement = (merged['references_past_mlx'] == merged['references_past_claude']).sum()
        print(f"\n=== COMPARISON WITH CLAUDE RESULTS ===")
        print(f"Matching prompts found: {len(merged)}")
        print(f"Agreement rate: {agreement/len(merged)*100:.2f}%")
        
        # Show disagreements
        disagreements = merged[merged['references_past_mlx'] != merged['references_past_claude']]
        if len(disagreements) > 0:
            print(f"\nDisagreements ({len(disagreements)}):")
            for _, row in disagreements.head(5).iterrows():
                print(f"\nPrompt: {row['prompt'][:80]}...")
                print(f"  MLX: {row['references_past_mlx']} | Claude: {row['references_past_claude']}")
    else:
        print("No matching prompts found between datasets")
else:
    print(f"No Claude results file found at {CLAUDE_RESULTS_FILE}")
    print("Skipping comparison.")

---

## 10. Full Dataset Processing (After Validation)

Once satisfied with validation results, use this section to process the full 10M dataset.

In [ ]:
# FULL DATASET CONFIGURATION
# Uncomment and run this section when ready to process the full dataset

"""
FULL_RESULTS_FILE = "data/results_mlx/full_results.csv"
CHECKPOINT_INTERVAL = 1000  # Save every N prompts

# Load full dataset
print("Loading full dataset...")
full_prompts_df = pd.read_csv('data/prompts.csv', usecols=[0])
full_prompts = full_prompts_df.iloc[:, 0].dropna().tolist()
full_prompts = [str(p).strip() for p in full_prompts if str(p).strip()]

print(f"Total prompts to process: {len(full_prompts):,}")

# Check for existing checkpoint
start_idx = 0
existing_results = []

if os.path.exists(FULL_RESULTS_FILE):
    existing_df = pd.read_csv(FULL_RESULTS_FILE)
    start_idx = len(existing_df)
    existing_results = existing_df.to_dict('records')
    print(f"Resuming from index {start_idx:,}")

# Process
results = existing_results.copy()

for i, prompt in enumerate(tqdm(full_prompts[start_idx:], desc="Processing full dataset")):
    try:
        classification, justification, gen_time = classify_prompt(prompt)
        results.append({
            'prompt': prompt,
            'references_past': classification,
            'justification': justification,
            'generation_time': gen_time
        })
    except Exception as e:
        results.append({
            'prompt': prompt,
            'references_past': 'error',
            'justification': str(e),
            'generation_time': 0
        })
    
    # Checkpoint
    if (start_idx + i + 1) % CHECKPOINT_INTERVAL == 0:
        pd.DataFrame(results).to_csv(FULL_RESULTS_FILE, index=False)
        logger.info(f"Checkpoint: {len(results):,} prompts processed")

# Final save
pd.DataFrame(results).to_csv(FULL_RESULTS_FILE, index=False)
print(f"Done! Processed {len(results):,} prompts")
"""

print("Full dataset processing is commented out.")
print("Uncomment the code above when ready to process all 10M prompts.")